In [1]:
# Cell 1: Import libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from clustering import (
    perform_kmeans_clustering,
    cluster_based_training,
    compare_global_vs_cluster_models
)

from utils import (
    plot_pca_2d,
    plot_tsne_2d,
    plot_class_distribution
)

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Cell 2: Load preprocessed data
print("Loading preprocessed data...")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Load training and test data
X_train = np.load('../data/X_train.npy',allow_pickle=True)
y_train = np.load('../data/y_train.npy',allow_pickle=True)
X_test = np.load('../data/X_test.npy',allow_pickle=True)

# Split training data into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Load feature names
with open('../data/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print("✓ Data loaded and split successfully!")
print(f"  Training samples: {X_train.shape[0]:,}")
print(f"  Validation samples: {X_val.shape[0]:,}")
print(f"  Test samples (no labels): {X_test.shape[0]:,}")
print(f"  Features: {X_train.shape[1]}")


Loading preprocessed data...
✓ Data loaded and split successfully!
  Training samples: 246,008
  Validation samples: 61,503
  Test samples (no labels): 48,744
  Features: 256


In [ ]:
# Cell 3: Perform K-Means clustering
print("\n" + "=" * 80)
print("PERFORMING K-MEANS CLUSTERING")
print("=" * 80)

n_clusters = 4
print(f"\nNumber of clusters: {n_clusters}")

kmeans_model, train_clusters, test_clusters, cluster_info = perform_kmeans_clustering(
    X_train, n_clusters=n_clusters
)

print(f"\n✓ Clustering completed!")
print(f"  Inertia: {kmeans_model.inertia_:.2f}")


PERFORMING K-MEANS CLUSTERING

Number of clusters: 4

Performing K-Means clustering with k=4...


In [ ]:
# Cell 4: Analyze cluster distributions
print("\n" + "=" * 80)
print("CLUSTER DISTRIBUTION ANALYSIS")
print("=" * 80)

print("\nTraining set cluster distribution:")
train_cluster_counts = pd.Series(train_clusters).value_counts().sort_index()
print(train_cluster_counts)

print("\nTest set cluster distribution:")
test_cluster_counts = pd.Series(test_clusters).value_counts().sort_index()
print(test_cluster_counts)

# Visualize cluster distribution
plot_cluster_distribution(
    train_clusters, 
    test_clusters,
    save_path='../outputs/cluster_distribution.png'
)

In [ ]:
# Cell 5: Cluster characteristics
print("\n" + "=" * 80)
print("CLUSTER CHARACTERISTICS")
print("=" * 80)

for cluster_id, info in cluster_info.items():
    print(f"\nCluster {cluster_id}:")
    print(f"  Size: {info['size']} samples")
    print(f"  Default rate: {info['default_rate']:.2%}")
    print(f"  Percentage of total: {info['percentage']:.2%}")

In [ ]:
# Cell 6: Target distribution per cluster
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for cluster_id in range(n_clusters):
    cluster_mask = train_clusters == cluster_id
    cluster_targets = y_train[cluster_mask]
    
    target_counts = pd.Series(cluster_targets).value_counts()
    
    axes[cluster_id].bar(['Repaid (0)', 'Default (1)'], 
                         [target_counts.get(0, 0), target_counts.get(1, 0)],
                         color=['#2ecc71', '#e74c3c'])
    axes[cluster_id].set_title(f'Cluster {cluster_id} - Target Distribution\n'
                               f'(Default Rate: {cluster_info[cluster_id]["default_rate"]:.2%})',
                               fontweight='bold')
    axes[cluster_id].set_ylabel('Count')
    
    # Add value labels
    for i, v in enumerate([target_counts.get(0, 0), target_counts.get(1, 0)]):
        axes[cluster_id].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/cluster_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7: PCA visualization of clusters
print("\n" + "=" * 80)
print("PCA VISUALIZATION")
print("=" * 80)

plot_pca_clusters(
    X_train, 
    train_clusters, 
    y_train,
    title='PCA Projection of Clusters (Training Data)',
    save_path='../outputs/pca_clusters.png'
)

In [ ]:
# Cell 8: t-SNE visualization of clusters
print("\n" + "=" * 80)
print("t-SNE VISUALIZATION")
print("=" * 80)

# Use subset for t-SNE (it's computationally expensive)
subset_size = min(5000, X_train.shape[0])
subset_indices = np.random.choice(X_train.shape[0], subset_size, replace=False)

plot_tsne_clusters(
    X_train[subset_indices], 
    train_clusters[subset_indices],
    y_train[subset_indices],
    title=f't-SNE Projection of Clusters (Sample: {subset_size} points)',
    save_path='../outputs/tsne_clusters.png'
)

In [ ]:
# Cell 9: Train models per cluster
print("\n" + "=" * 80)
print("TRAINING PER-CLUSTER MODELS")
print("=" * 80)

cluster_models = train_per_cluster_models(
    X_train, y_train, train_clusters,
    X_test, y_test, test_clusters,
    n_clusters=n_clusters
)

print("\n✓ Per-cluster models trained!")

In [ ]:
# Cell 10: Evaluate cluster performance
print("\n" + "=" * 80)
print("EVALUATING CLUSTER-BASED MODELS")
print("=" * 80)

cluster_metrics = evaluate_cluster_performance(
    cluster_models,
    X_test,
    y_test,
    test_clusters,
    n_clusters=n_clusters
)

In [ ]:
# Cell 11: Display cluster metrics
print("\n" + "=" * 80)
print("PER-CLUSTER PERFORMANCE METRICS")
print("=" * 80)

cluster_results = []
for cluster_id, metrics in cluster_metrics.items():
    print(f"\nCluster {cluster_id}:")
    print(f"  Samples: {metrics['n_samples']}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")
    print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    
    cluster_results.append({
        'Cluster': cluster_id,
        'Samples': metrics['n_samples'],
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1'],
        'ROC-AUC': metrics['roc_auc']
    })

# Create DataFrame
cluster_df = pd.DataFrame(cluster_results)
print("\n" + "=" * 80)
print("CLUSTER METRICS SUMMARY")
print("=" * 80)
print(cluster_df.to_string(index=False))

# Save cluster metrics
cluster_df.to_csv('../outputs/cluster_metrics.csv', index=False)
print("\n✓ Cluster metrics saved to outputs/cluster_metrics.csv")

In [ ]:
# Cell 12: Visualize cluster performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    bars = ax.bar(cluster_df['Cluster'], cluster_df[metric], color=colors)
    ax.set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} per Cluster', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.set_xticks(cluster_df['Cluster'])
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/cluster_performance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 13: Overall cluster-based approach performance
print("\n" + "=" * 80)
print("OVERALL CLUSTER-BASED PERFORMANCE")
print("=" * 80)

# Calculate weighted average metrics
total_samples = cluster_df['Samples'].sum()
weighted_accuracy = (cluster_df['Accuracy'] * cluster_df['Samples']).sum() / total_samples
weighted_precision = (cluster_df['Precision'] * cluster_df['Samples']).sum() / total_samples
weighted_recall = (cluster_df['Recall'] * cluster_df['Samples']).sum() / total_samples
weighted_f1 = (cluster_df['F1-Score'] * cluster_df['Samples']).sum() / total_samples
weighted_roc_auc = (cluster_df['ROC-AUC'] * cluster_df['Samples']).sum() / total_samples

print(f"\nWeighted Average Performance (across all clusters):")
print(f"  Accuracy:  {weighted_accuracy:.4f}")
print(f"  Precision: {weighted_precision:.4f}")
print(f"  Recall:    {weighted_recall:.4f}")
print(f"  F1-Score:  {weighted_f1:.4f}")
print(f"  ROC-AUC:   {weighted_roc_auc:.4f}")

In [ ]:
# Cell 14: Compare with baseline (load from previous notebook results)
print("\n" + "=" * 80)
print("COMPARISON: CLUSTER-BASED vs BASELINE MODELS")
print("=" * 80)

# Load baseline comparison
try:
    baseline_df = pd.read_csv('../outputs/model_comparison.csv')
    
    print("\nBaseline Models Performance:")
    print(baseline_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].to_string(index=False))
    
    best_baseline_f1 = baseline_df['F1-Score'].max()
    best_baseline_model = baseline_df.loc[baseline_df['F1-Score'].idxmax(), 'Model']
    
    print(f"\n📊 Best Baseline Model: {best_baseline_model}")
    print(f"   F1-Score: {best_baseline_f1:.4f}")
    
    print(f"\n📊 Cluster-Based Approach:")
    print(f"   Weighted F1-Score: {weighted_f1:.4f}")
    
    improvement = ((weighted_f1 - best_baseline_f1) / best_baseline_f1) * 100
    
    if improvement > 0:
        print(f"\n✓ Cluster-based approach shows {improvement:.2f}% improvement over best baseline!")
    else:
        print(f"\n⚠ Cluster-based approach is {abs(improvement):.2f}% lower than best baseline")
        print("  Consider: different clustering algorithms, optimal k, feature selection")
    
except FileNotFoundError:
    print("\n⚠ Baseline model comparison not found. Run 02_Model_Training.ipynb first.")

In [ ]:
# Cell 15: Cluster insights
print("\n" + "=" * 80)
print("CLUSTER INSIGHTS & RECOMMENDATIONS")
print("=" * 80)

# Find best and worst performing clusters
best_cluster = cluster_df.loc[cluster_df['F1-Score'].idxmax()]
worst_cluster = cluster_df.loc[cluster_df['F1-Score'].idxmin()]

print(f"\n🏆 Best Performing Cluster: {int(best_cluster['Cluster'])}")
print(f"   F1-Score: {best_cluster['F1-Score']:.4f}")
print(f"   Default Rate: {cluster_info[int(best_cluster['Cluster'])]['default_rate']:.2%}")

print(f"\n⚠ Lowest Performing Cluster: {int(worst_cluster['Cluster'])}")
print(f"   F1-Score: {worst_cluster['F1-Score']:.4f}")
print(f"   Default Rate: {cluster_info[int(worst_cluster['Cluster'])]['default_rate']:.2%}")

print(f"\n💡 Key Insights:")
print(f"   • Clusters with higher default rates may need specialized models")
print(f"   • Cluster size and class balance affect model performance")
print(f"   • Per-cluster models can capture group-specific patterns")

In [ ]:
# Cell 16: Save all results
print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Save overall cluster-based performance
overall_results = pd.DataFrame([{
    'Approach': 'Cluster-Based (Weighted Avg)',
    'Accuracy': weighted_accuracy,
    'Precision': weighted_precision,
    'Recall': weighted_recall,
    'F1-Score': weighted_f1,
    'ROC-AUC': weighted_roc_auc
}])

overall_results.to_csv('../outputs/cluster_based_overall.csv', index=False)
print("✓ Overall cluster-based results saved")

# Save cluster info
cluster_info_df = pd.DataFrame([
    {
        'Cluster': k,
        'Size': v['size'],
        'Percentage': v['percentage'],
        'Default_Rate': v['default_rate']
    }
    for k, v in cluster_info.items()
])
cluster_info_df.to_csv('../outputs/cluster_info.csv', index=False)
print("✓ Cluster information saved")

In [ ]:
# Cell 17: Summary and conclusions
print("\n" + "=" * 80)
print("CLUSTERING ANALYSIS COMPLETE")
print("=" * 80)
print(f"""
Summary:
--------
✓ Performed K-Means clustering with {n_clusters} clusters
✓ Trained separate models for each cluster
✓ Evaluated per-cluster and overall performance
✓ Compared with baseline models

Generated Outputs:
------------------
• PCA and t-SNE cluster visualizations
• Cluster distribution plots
• Per-cluster performance metrics
• Cluster-based vs baseline comparison

Key Findings:
-------------
• Total clusters: {n_clusters}
• Weighted Avg F1-Score: {weighted_f1:.4f}
• Best performing cluster: {int(best_cluster['Cluster'])} (F1: {best_cluster['F1-Score']:.4f})
• Worst performing cluster: {int(worst_cluster['Cluster'])} (F1: {worst_cluster['F1-Score']:.4f})

Recommendations:
----------------
1. Experiment with different numbers of clusters (elbow method)
2. Try other clustering algorithms (DBSCAN, Hierarchical)
3. Perform feature selection for each cluster
4. Use ensemble methods combining cluster-based predictions
5. Investigate cluster characteristics for business insights
""")